In [ ]:
import os
import re
import time
import unicodedata

import pymongo
import numpy as np
import pandas as pd
import torch
from pyvi import ViTokenizer
from transformers import AutoTokenizer, RobertaForSequenceClassification
from tqdm import tqdm
from collections import Counter

In [6]:
from google.colab import drive
drive.mount('/content/drive')
%cd "/content/drive/MyDrive/Fpt Sentiment Analysis"

Mounted at /content/drive
/content/drive/MyDrive/Fpt Sentiment Analysis


In [ ]:
CONNECTION_STRING = "mongodb+srv://<USERNAME>:<PASSWORD>@cluster0.../test"

In [ ]:
client = pymongo.MongoClient(CONNECTION_STRING)
db = client['Data_Engineer']
collection = db['Comments']

In [ ]:
total = collection.count_documents({})
print('Tổng số comment:', total)

Tổng số comment: 885169


In [ ]:
cursor = collection.find()

In [ ]:
list_comments = list(cursor)

In [ ]:
df = pd.DataFrame(list_comments)
print(f'Kích thước bảng dữ liệu: {df.shape[0]} dòng và {df.shape[1]} cột')

Kích thước bảng dữ liệu: 885169 dòng và 21 cột


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 885169 entries, 0 to 885168
Data columns (total 21 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   _id             885169 non-null  object
 1   ranking         885169 non-null  int64 
 2   layer           885169 non-null  int64 
 3   user_id         885169 non-null  int64 
 4   like            885169 non-null  int64 
 5   user_email      885169 non-null  object
 6   comment_on      885169 non-null  object
 7   ip              885169 non-null  object
 8   user_fullname   885169 non-null  object
 9   publish_status  885169 non-null  int64 
 10  object_id       885169 non-null  object
 11  content         885169 non-null  object
 12  reviewer        885169 non-null  object
 13  report_count    885169 non-null  int64 
 14  review_status   885169 non-null  int64 
 15  timestamp       885169 non-null  int64 
 16  device          885169 non-null  object
 17  report_reason   885169 non-nu

In [ ]:
df.head(3)

,_id,ranking,layer,user_id,like,user_email,comment_on,ip,user_fullname,publish_status,...,content,reviewer,report_count,review_status,timestamp,device,report_reason,dislike,comment_status,device_id
0,555466761a77ce5d65ea062a,0,0,12,1,hoaianh2210@gmail.com,event,null,Nguyễn Hoài Anh,1,...,"Chương trình \ Tuyệt cú mèo\ luôn, chờ lâu rồi...",gianpt,0,1,1431595030,web-playfpt,[],0,0,null
1,555457bf1a77ce5d65ea0617,0,0,130,5,gian.phan@gmail.com,event,null,Phan Thanh Gian,1,...,Chờ đợi chương trình này đã lâu! Hẹn mọi người...,gianpt,0,1,1431591263,web-playfpt,[],0,0,null
2,555413001a77ce5d65ea0609,0,0,527201,0,cloudytang@gmail.com,vod,42.116.8.164,Đàm Ngọc Vân,0,...,comment cai,vuluc88,0,1,1431573662,android,[],0,2,983b5151a74de256


In [ ]:
df = df.drop(['ranking', 'layer', 'user_email','user_fullname','report_count','review_status', 'report_reason', 'comment_status','device_id'], axis =1)

In [ ]:
df.head(3)

,_id,user_id,like,comment_on,ip,publish_status,object_id,content,reviewer,timestamp,device,dislike
0,555466761a77ce5d65ea062a,12,1,event,null,1,552b2f0817dc135c89d7d6a2,"Chương trình \ Tuyệt cú mèo\ luôn, chờ lâu rồi...",gianpt,1431595030,web-playfpt,0
1,555457bf1a77ce5d65ea0617,130,5,event,null,1,552b2f0817dc135c89d7d6a2,Chờ đợi chương trình này đã lâu! Hẹn mọi người...,gianpt,1431591263,web-playfpt,0
2,555413001a77ce5d65ea0609,527201,0,vod,42.116.8.164,0,5453639f17dc1371f538a4cd,comment cai,vuluc88,1431573662,android,0


In [ ]:
df['datetime'] = pd.to_datetime(df['timestamp'], unit = 's')
df.head(1)

,_id,user_id,like,comment_on,ip,publish_status,object_id,content,reviewer,timestamp,device,dislike,datetime
0,555466761a77ce5d65ea062a,12,1,event,null,1,552b2f0817dc135c89d7d6a2,"Chương trình \ Tuyệt cú mèo\ luôn, chờ lâu rồi...",gianpt,1431595030,web-playfpt,0,2015-05-14 09:17:10


In [ ]:
df['datetime'].agg(['min','max'])

In [ ]:
df['date'] = df['datetime'].dt.date
df['dayofweek'] = df['datetime'].dt.weekday
df['hour'] = df['datetime'].dt.hour
df.head(1)

,_id,user_id,like,comment_on,ip,publish_status,object_id,content,reviewer,timestamp,device,dislike,datetime,date,dayofweek,hour
0,555466761a77ce5d65ea062a,12,1,event,null,1,552b2f0817dc135c89d7d6a2,"Chương trình \ Tuyệt cú mèo\ luôn, chờ lâu rồi...",gianpt,1431595030,web-playfpt,0,2015-05-14 09:17:10,2015-05-14,3,9


In [ ]:
df['dayofweek'].unique()

array([3, 2, 4, 5, 6, 0, 1], dtype=int32)

In [ ]:
df['dayofweek'] = df['dayofweek'].apply(lambda x : 'Thứ 2' if x == 0
                                                            else ('Thứ 3' if x == 1
                                                            else ('Thứ 4' if x == 2
                                                            else ('Thứ 5' if x == 3
                                                            else ('Thứ 6' if x == 4
                                                            else ('Thứ 7' if x == 5
                                                            else 'Chủ nhật'))))))

In [ ]:
df['comment_on'].unique()

array(['event', 'vod', 'channel', 'highlight', '3', 'comment',
       'hightlight', 'phim-bo', 'livetv', 'tvshow', 'anime', 'thieu-nhi',
       'phim-Vip', 'general', 'sport', 'fadio', 'the-thao',
       'ngoai-hang-anh', 'phim-le', 'TV show', 'phim-le-old',
       'truyen audio - VOD', 'tet-2017', 'esports', 'phim-gia-dinh',
       'phim-vip-cu', 'fifa-u20-world-cup', 'glee-viet-nam',
       'asean-basketball-league', 'tv-show', 'tet-2018', '', 'orionmedia',
       'fifa-world-cup-2018', 'tet-2019', 'hoc-online', 'tet-2021'],
      dtype=object)

In [ ]:
unknown = ['3', 'comment', 'general', 'orionmedia', 'fadio', 'hoc-online', '']

for tag in unknown:
    sub = df[df['comment_on'] == tag]
    print(f'Comment_on: {tag}')
    print(sub['content'].sample(min(60, len(sub)), random_state=1).tolist())

Comment_on: 3
['https://www.facebook.com/xiu.reno.3/media_set?set=a.934029943333897']
Comment_on: comment
['phim như cc', 'Phim xem buồn ngủ thật đó. Suzy diễn lố quá, nhìn không tự nhiên, tình tiết không thấy gì hấp dẫn.', 'Bản HTV2 LỒNG hay hơn nhiêug', 'Chán ghê vô xem ko đc, chẳng biết bị gì ???', 'Kthuc s v mn', 'Hóng tập 15 quá', 'Hay qua ko biết chung', 'ui trời ơi, mới nghe op là biết ngay là Oldcodex roy ahihi hay quá đê,', '123456789', 'Dau pai anime dau mak thag nao noi j', 'Kết mở ! Ghét thật', 'hay quá đy', 'thich nhat chuon chuon giay!!!', 'Trừ cái Faded ra thì còn lại không nuốt nổi', 'Tap 277 zombie', 'Khi nào mới thuyết minh 15 16 17 18', 'aaaaaaaaaaaaaaaaaaaaa', '😎😎😎😎😠😠😠😠☺☺☺☺', 'cái poster thấy iron man là biết phim nhảm rồi, chẳng biết có that ko, chưa xem', 'Phim hay', 'http://ouo.io/Po4QON', 'thích FPT rồi  á, cập Nhật phim nhanh quá trời luôn :v', 'Fim này cuối cùng chắc chớt wơớt....', 'đam mỹ mà hay hơn ngôn tình @@~', 'doc hay hot', 'Doc gioi thieu cung hap da

In [ ]:
sports = ['sport', 'the-thao', 'ngoai-hang-anh', 'highlight', 'hightlight', 'fifa-world-cup-2018', 'fifa-u20-world-cup', 'esports', 'asean-basketball-league']
anime_kids = ['anime', 'thieu-nhi']
live_event = ['livetv', 'channel', 'event']
tv_show = ['tvshow', 'TV show', 'tv-show', 'general', 'orionmedia', 'tet-2017', 'tet-2018', 'tet-2019', 'tet-2021', 'fadio', 'truyen audio - VOD', 'hoc-online']

df['category'] = df['comment_on'].apply(lambda x : 'Sport' if x in sports
                                                            else ('Anime & kids' if x in anime_kids
                                                            else ('Channel & event' if x in live_event
                                                            else ('TV show & Audio' if x in tv_show
                                                            else 'Movies & Series'))))
df['category'].value_counts()

,count
category,
Movies & Series,655367
Sport,183684
Channel & event,44006
Anime & kids,1789
TV show & Audio,323


In [ ]:
empty_count = (df['content'].fillna('').str.strip() == '').sum()
empty_count

np.int64(5)

In [ ]:
#bỏ những content rỗng
df = df[df['content'].fillna('').str.strip() != '']

In [ ]:
# test
sample_text = "anh trung ruồi trông buồn cười thật"
print("Sau khi tách từ:", ViTokenizer.tokenize(sample_text))

Sau khi tách từ: anh trung ruồi trông buồn_cười thật


In [ ]:
# Gộp toàn bộ bình luận và chuyển về chữ thường
all_text = " ".join(df['content'].dropna().astype(str).str.lower())

# Tách tất cả các từ (chỉ lấy chữ cái)
words = re.findall(r'\b[a-zàáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ]+\b', all_text)

# ập hợp tất cả nguyên âm tiếng Việt
vowels = set("aăâeêioôơuưyàáảãạằắẳẵặầấẩẫậèéẻẽẹềếểễệìíỉĩịòóỏõọồốổỗộờớởỡợùúủũụừứửữựỳýỷỹỵ")

# Lọc các từ ngắn (1-5 ký tự) KHÔNG chứa nguyên âm nào
consonant_only_words = [w for w in words if not any(ch in vowels for ch in w) and len(w) <= 5]

# Đếm Top 50 từ xuất hiện nhiều nhất
top_consonant_abbr = Counter(consonant_only_words).most_common(200)

print("=== TOP 200 TỪ VIẾT TẮT (KHÔNG NGUYÊN ÂM) ===")
for word, count in top_consonant_abbr:
    print(f"{word:<8} : {count:,} lần")

=== TOP 200 TỪ VIẾT TẮT (KHÔNG NGUYÊN ÂM) ===
k        : 52,686 lần
t        : 25,841 lần
đc       : 24,951 lần
r        : 22,420 lần
ng       : 20,082 lần
v        : 19,790 lần
mn       : 15,681 lần
vs       : 15,619 lần
n        : 15,272 lần
j        : 13,839 lần
vl       : 13,039 lần
m        : 12,953 lần
dc       : 12,743 lần
vn       : 10,285 lần
z        : 9,958 lần
fpt      : 9,391 lần
h        : 8,677 lần
c        : 8,464 lần
s        : 7,223 lần
blv      : 7,009 lần
nn       : 6,605 lần
p        : 6,516 lần
cm       : 6,494 lần
hp       : 6,412 lần
vc       : 4,887 lần
ms       : 4,883 lần
bn       : 4,473 lần
cc       : 4,332 lần
mk       : 4,309 lần
đm       : 4,241 lần
nh       : 4,238 lần
b        : 4,099 lần
cx       : 3,626 lần
vcl      : 3,595 lần
cmt      : 3,409 lần
dm       : 2,955 lần
kh       : 2,920 lần
bt       : 2,913 lần
lm       : 2,780 lần
tv       : 2,674 lần
ch       : 2,631 lần
hk       : 2,570 lần
nv       : 2,566 lần
rm       : 2,473 lần
sm       : 2,403

In [ ]:
# Lọc comment viết tắt đoán nghĩa
dv_comments = df[df['content'].str.lower().str.contains(r'\bnhg\b', regex=True, na=False)]['content']  #thay từ để đoán nghĩa
print(f"Tổng số comment chứa từ 'dv': {len(dv_comments):,} dòng\n")
samples = dv_comments.sample(10, random_state=42).tolist()
for i, s in enumerate(samples, 1):
    print(f"{i:>2}. {s}")

Tổng số comment chứa từ 'dv': 331 dòng

 1. Phim hay thật, rất lôi cuốn... nhg để ý mới thấy những đoạn quay 2 nhân vật chính đều bị xóa phông, ảo, k thực.. đọc báo mới biết thì ra 2 nhân vật chính này bê bối trg sự nghiệp khi đóng bộ phim này. Mình xem thấy cốt truyện rất hay, nhân vật đẹp nhg vẫn cảm thấy có gì đó k tôn trọng khán giả vì cứ nghĩ đến 2 nhân vật toàn nhờ diễn viên đóng thế và 2 nv chính chủ yếu đóng dưới phông nền xanh xong đem cắt ghép hình ảnh vào... Ánh mắt CHL nhìn AB cũng k cảm xúc gì cả, cô AB thì diễn cũng k hay nhg công nhận nv chính đẹp quá.
 2. nữ 9 mạnh mẽ, nhg lại ko mưu trí. quyết đoán sát phạt những lại để sự giận dữ lấn át.
 3. M chỉ nghĩ là do cách đào tạo idol ở NB vs HQ khác nhau thôi. VD như TTS NB cũng đc đào tạo nhảy nhót thường xuyên như ở HQ thì họ cũng sẽ giỏi thôi. Mà đúng là có nhg ng mà m cứ nghĩ họ bt nhg lại có sức hút công chúng, nên nổi tiếng là điều hết sức bt :P ko có gì phải băn khoăn cả.
 4. bạn Băng Nhi dù Dương tử ko đẹp dc bằng Cúc

In [ ]:
# Từ điển chuẩn hóa từ viết tắt
ABBR_DICT = {
    # 1. Phủ định & Trạng thái
    'k': 'không', 'ko': 'không', 'kh': 'không', 'hk': 'không', 'kg': 'không', 'hong': 'không', 'hem': 'không',
    'kp': 'không phải', 'kb': 'không biết', 'ks': 'không sao', 'kbh': 'không bao giờ',
    'đc': 'được', 'dc': 'được', 'dk': 'được', 'đk': 'được', 'dx': 'được', 'đx': 'được',
    'r': 'rồi', 'roi': 'rồi', 'cmnr': 'rồi',
    'cx': 'cũng', 'cg': 'cũng', 'ms': 'mới', 'bt': 'bình thường', 'bth': 'bình thường',
    'trc': 'trước', 'nx': 'nữa', 'đg': 'đang', 'ln': 'luôn', 'nhg': 'nhưng', 'trg': 'trong',

    # 2. Hỏi đáp, Thời gian & Giao tiếp
    'v': 'vậy', 'z': 'vậy', 'j': 'gì', 'g': 'gì', 'cj': 'cái gì',
    'vs': 'với', 's': 'sao', 'lm': 'làm', 'ns': 'nói', 'nc': 'nói chuyện',
    'h': 'giờ', 'bh': 'bây giờ', 'hn': 'hôm nay', 'hm': 'hôm',
    'ntn': 'như thế nào', 'ltn': 'như thế nào', 'htn': 'như thế nào',
    'lh': 'liên hệ', 'nt': 'nhắn tin', 'tl': 'trả lời',
    'bk': 'biết', 'thk': 'thích', 'xm': 'xem', 'hc': 'học',
    'tks': 'cảm ơn', 'hmtks': 'cảm ơn', 'xl': 'xin lỗi',

    # 3. Xưng hô & Con người
    't': 'tôi', 'm': 'mày', 'ng': 'người', 'mn': 'mọi người', 'mng': 'mọi người',
    'bn': 'bạn', 'b': 'bạn', 'mk': 'mình', 'mh': 'mình', 'mjh': 'mình',
    'ck': 'chồng', 'vk': 'vợ', 'thg': 'thằng', 'ae': 'anh em',

    # 4. Phim ảnh, Bóng đá & FPT Play
    'ad': 'admin', 'adm': 'admin', 'blv': 'bình luận viên', 'hlv': 'huấn luyện viên',
    'cmt': 'bình luận', 'bl': 'bình luận', 'tv': 'tivi', 'dt': 'điện thoại', 'đt': 'điện thoại',
    'nv': 'nhân vật', 'dv': 'diễn viên', 'dvd': 'diễn viên', 'rm': 'running man',
    'pd': 'phụ đề', 'tm': 'thuyết minh', 'ltt': 'lồng tiếng', 'qc': 'quảng cáo',
    'mc': 'người dẫn chương trình', 'tk': 'tài khoản', 'ss': 'mùa', 'tp': 'tập',
    'ct': 'chương trình', 'nd': 'nội dung', 'tg': 'thời gian', 'hd': 'sắc nét',
    'phjm': 'phim', 'fbt': 'fpt', 'fb': 'facebook', 'gg': 'google',

    # 5. Quốc gia & Giải đấu
    'vn': 'việt nam', 'nn': 'nước ngoài', 'tbn': 'tây ban nha', 'tq': 'trung quốc',
    'hq': 'hàn quốc', 'bdn': 'bồ đào nha', 'bđn': 'bồ đào nha', 'nb': 'nhật bản',
    'wc': 'world cup', 'hcv': 'huy chương vàng',

    # 6. Cảm xúc, Khen & Chửi thề
    'hp': 'hạnh phúc', 'dth': 'dễ thương', 'đz': 'đẹp trai',
    'kk': 'haha', 'kkk': 'haha', 'kkkk': 'haha', 'kkkkk': 'haha', 'hjhj': 'haha', 'hj': 'haha', 'hh': 'haha',
    'vl': 'quá', 'vc': 'quá', 'vcl': 'quá', 'vkl': 'quá', 'vcc': 'quá', 'vch': 'quá', 'vđ': 'quá',
    'đm': 'bực mình', 'dm': 'bực mình', 'dkm': 'bực mình', 'đcm': 'bực mình',
    'dcm': 'bực mình', 'cmm': 'bực mình', 'đkm': 'bực mình', 'cmn': 'bực mình',
    'cc': 'tồi tệ', 'cl': 'tồi tệ', 'lz': 'tồi tệ', 'qq': 'tồi tệ', 'ml': 'tồi tệ', 'rr': 'rẻ rách',
    'wtf': 'vô lý', 'sml': 'thảm hại'
}

def clean_vietnamese_text(text):
    if not text or not isinstance(text, str):
        return ""

    # Gom các chữ bị tách dấu về chữ tiếng Việt chuẩn rồi chuyển chữ thường
    text = unicodedata.normalize('NFC', text).lower()

    # Rút gọn chữ cái lặp (Phải làm ở đây để kooooo -> ko, dccccccc -> dc)
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    # Chỉ giữ lại chữ cái tiếng việt và khoảng trắng
    text = re.sub(r"[^a-z0-9àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ\s]", " ", text)

    # Tách từng từ và tra từ điển ABBR_DICT để thay thế từ viết tắt
    words = [ABBR_DICT.get(w, w) for w in text.split()]
    text = " ".join(words)

    # Nối từ ghép tiếng Việt bằng pyvi cho PhoBERT
    if text:
        text = ViTokenizer.tokenize(text)

    return text

In [ ]:
test_sentences = [
    "phim như cc, xem k đc j hết vl",
    "blv ns chuyện dth vcl, mn xem thử đi k phí tg đâu",
    "đm mạng lag vkl k xm đk tập ms",
    "rm tập hn hay vl kkk cảm ơn fbt"
]

for s in test_sentences:
    print("Gốc :", s)
    print("Sạch:", clean_vietnamese_text(s))
    print('')

Gốc : phim như cc, xem k đc j hết vl
Sạch: phim như tồi_tệ xem không được gì hết quá

Gốc : blv ns chuyện dth vcl, mn xem thử đi k phí tg đâu
Sạch: bình_luận_viên nói_chuyện dễ_thương quá mọi người xem thử đi không phí thời_gian đâu

Gốc : đm mạng lag vkl k xm đk tập ms
Sạch: bực_mình mạng lag quá không xem được tập mới

Gốc : rm tập hn hay vl kkk cảm ơn fbt
Sạch: running man tập hôm_nay hay quá không cảm_ơn fpt



In [ ]:
tqdm.pandas(desc="Đang làm sạch & tách từ")

df['content_clean'] = df['content'].progress_apply(clean_vietnamese_text)

Đang làm sạch & tách từ: 100%|██████████| 885164/885164 [02:23<00:00, 6161.14it/s]


In [ ]:
df = df[df['content_clean'] != ""].reset_index(drop=True)

LƯU ĐỂ PARQUET CONTENT ĐÃ CLEAN SAU CHỈ CẦN BẬT LÊN ĐỂ CHẠY TIẾP CÁC BATCH SENTIMENT CÒN LẠI

In [ ]:
# Ép kiểu _id và object_id sang dạng string thông thường
df['_id'] = df['_id'].astype(str)
if 'object_id' in df.columns:
    df['object_id'] = df['object_id'].astype(str)
# Lưu xuống Drive dạng parquet
df.to_parquet("/content/drive/MyDrive/Fpt Sentiment Analysis/df_cleaned.parquet", index=False)

In [ ]:
df = pd.read_parquet("/content/drive/MyDrive/Fpt Sentiment Analysis/df_cleaned.parquet")
print("Đã load xong:", df.shape)

Đã load xong: (880009, 18)


In [ ]:
df.head(5)

,_id,user_id,like,comment_on,ip,publish_status,object_id,content,reviewer,timestamp,device,dislike,datetime,date,dayofweek,hour,category,content_clean
0,555466761a77ce5d65ea062a,12,1,event,null,1,552b2f0817dc135c89d7d6a2,"Chương trình \ Tuyệt cú mèo\ luôn, chờ lâu rồi...",gianpt,1431595030,web-playfpt,0,2015-05-14 09:17:10,2015-05-14,Thứ 5,9,Channel & event,chương_trình tuyệt cú_mèo luôn chờ lâu rồi mới...
1,555457bf1a77ce5d65ea0617,130,5,event,null,1,552b2f0817dc135c89d7d6a2,Chờ đợi chương trình này đã lâu! Hẹn mọi người...,gianpt,1431591263,web-playfpt,0,2015-05-14 08:14:23,2015-05-14,Thứ 5,8,Channel & event,chờ_đợi chương_trình này đã lâu hẹn mọi người ...
2,555413001a77ce5d65ea0609,527201,0,vod,42.116.8.164,0,5453639f17dc1371f538a4cd,comment cai,vuluc88,1431573662,android,0,2015-05-14 03:21:02,2015-05-14,Thứ 5,3,Movies & Series,comment_cai
3,555357f71a77ce5d65ea0606,424467,1,vod,null,1,551a43b417dc135c58f3c691,"lồng tiếng mà như tập đọc, nghe không nuốt nổi...",vuluc88,1431525776,web-playfpt,0,2015-05-13 14:02:56,2015-05-13,Thứ 4,14,Movies & Series,lồng_tiếng mà như tập đọc nghe không nuốt nổi ...
4,555491181a77ce5d65ea0636,805,12,vod,42.117.66.148,1,5549cace17dc132a0cee1801,Phim rất hay ;),vuluc88,1431605947,android,0,2015-05-14 12:19:07,2015-05-14,Thứ 5,12,Movies & Series,phim rất hay


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [ ]:
# Load pre_trained model
model = RobertaForSequenceClassification.from_pretrained("wonrax/phobert-base-vietnamese-sentiment")
# Load tokeniner từ model
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base", use_fast=False)

# Đưa model lên gpu
model.to(device)

config.json:   0%|          | 0.00/999 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  540MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  540MB            

model.safetensors: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(64001, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(258, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
             

In [ ]:
sentence = 'Anh Dũng đẹp trai quá'

input_ids = torch.tensor([tokenizer.encode(sentence, add_special_tokens=True)]).to(device)

with torch.no_grad():
    out = model(input_ids)
    print(pd.Series(out.logits.softmax(dim=-1).tolist()[0], index = ['NEG', 'POS', 'NEU']))

NEG    0.001730
POS    0.989982
NEU    0.008288
dtype: float64


In [ ]:
# Danh sách các nguyên âm tiếng Việt
VOWELS = "aeiouyàáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵ"

def sentiment(sentence):
    try:
        # Lọc bỏ từ vô nghĩa
        words = sentence.lower().replace('_', ' ').split()
        if any(len(w) >= 10 and w.isalpha() for w in words):
            return 2
        if not any(c in VOWELS for c in sentence.lower()):
            return 2

        input_ids = tokenizer.encode(sentence, truncation=True, max_length=200)
        input_ids = torch.tensor([input_ids]).to(device)

        with torch.no_grad():
            out = model(input_ids)
            probs = out.logits.softmax(dim=-1).tolist()[0]
            neg, pos, neu = probs[0], probs[1], probs[2]

            # threshold
            if pos >= 0.1 and pos > neg * 1.5:
                return 1
            elif neg >= 0.1 and neg > pos * 1.5:
                return 0
            else:
                return 2
    except Exception:
        return 2

In [ ]:
start_time = time.time()
tqdm.pandas(desc="Đang phân tích sentiment")
# Lấy thử 1000 dòng đầu tiên
test = df[:1000].copy()
test['sentiment'] = test['content_clean'].progress_apply(sentiment)
print("--- Chạy xong trong: %s giây ---" % round(time.time() - start_time, 2))

Đang phân tích sentiment: 100%|██████████| 1000/1000 [00:08<00:00, 116.50it/s]

--- Chạy xong trong: 8.59 giây ---


In [ ]:
test = test[['content_clean', 'sentiment']]
test

,content_clean,sentiment
0,chương_trình tuyệt cú_mèo luôn chờ lâu rồi mới...,1
1,chờ_đợi chương_trình này đã lâu hẹn mọi người ...,1
2,comment_cai,1
3,lồng_tiếng mà như tập đọc nghe không nuốt nổi ...,0
4,phim rất hay,1
...,...,...
995,phim nào coi cũng bị giật hết admin sửa lại giùm,0
996,mình có link đây ai muốn xem thì inbox faceboo...,0
997,45 tap_nhe,1
998,mau mau co tap 45 46 47 nka,0


In [ ]:
test['sentiment'].value_counts()

,count
sentiment,
1,467
0,317
2,216


In [ ]:
test2 = test[test['sentiment'] == 1]
test2.tail(10)

In [ ]:
file_name = "fpt_full_sentiment.csv"

# Nếu đã có file thì đếm số dòng để chạy tiếp, chưa có thì bắt đầu từ 0
start_row = len(pd.read_csv(file_name)) if os.path.exists(file_name) else 0
print("Bắt đầu chạy từ dòng:", start_row)

# Cắt từng batch 50.000 dòng để chạy và lưu nối tiếp vào file
for i in range(start_row, len(df), 50000):
    batch = df[i : i + 50000].copy()

    # Cập nhật tên thanh tiến trình cho từng Batch
    tqdm.pandas(desc=f"Batch {i//50000 + 1}")
    batch['sentiment'] = batch['content_clean'].progress_apply(sentiment)

    # Xóa cột content_clean cho nhẹ file rồi mới lưu vào CSV
    batch = batch.drop(columns=['content_clean'])
    batch.to_csv(file_name, mode='a', header=(i == 0), index=False)

    # 10 dòng đầu và 10 dòng cuối của mẻ vừa xong
    print(f"\n=== ĐÃ LƯU XONG ĐẾN DÒNG: {i + len(batch):,} ===")
    print(batch['sentiment'].value_counts())
    print()
    print("\n--- 10 DÒNG ĐẦU ---")
    print(batch[['content', 'sentiment']].head(10))
    print()
    print("\n--- 10 DÒNG CUỐI ---")
    print(batch[['content', 'sentiment']].tail(10))
    print()
    print()

Bắt đầu chạy từ dòng: 700000


Batch 15: 100%|██████████| 50000/50000 [07:35<00:00, 109.80it/s]



=== ĐÃ LƯU XONG ĐẾN DÒNG: 750,000 ===
sentiment
1    22799
0    19211
2     7990
Name: count, dtype: int64


--- 10 DÒNG ĐẦU ---
                                                  content  sentiment
700000               nữ chính có răng khểnh nhìn xinh quá          1
700001  nói về bệnh hoạn thì nên coi i saw devil rồi m...          0
700002                                  sau ch thay sub z          0
700003                   hết rồi ak có một chút tiết nuối          1
700004               phim hay quá dù ngàn năm vẫn chờ đợi          1
700005  nữ chính xinh từ trong phim đến ngoài đời ẻm đ...          1
700006  Bộ này 26 tập nha, fpt nó đề sai ý chứ sau này...          2
700007           xem bên danet.vn full show lâu r mấy bạn          1
700008                                chừng nào có Phần 3          1
700009                                     Hóng quá ad oi          1


--- 10 DÒNG CUỐI ---
                                                  content  sentiment
749990  Khánh Linh 

Batch 16: 100%|██████████| 50000/50000 [07:29<00:00, 111.30it/s]



=== ĐÃ LƯU XONG ĐẾN DÒNG: 800,000 ===
sentiment
1    23121
0    18817
2     8062
Name: count, dtype: int64


--- 10 DÒNG ĐẦU ---
                                                  content  sentiment
750000                    haizzz hết  p2 rồi ở đây vẫn p1          0
750001                                       kết buồn quá          0
750002  [TẬP 36 THE END] Vậy là chặn đường 36 tập phim...          1
750003      mn lên động phim trên gg ấy~ phim ra full rồi          1
750004  mn có cảm thấy nữ chính có nét giống trần kiều...          1
750005   Đúng là tình yêu đẹp chỉ có trong phim mà thôi.!          1
750006                                        hôm nay tlo          1
750007                     Vũ kỳ xinh quá ❤ cute phết nhỉ          1
750008                 15A bị lỗi ạ nó là trailer kìa :))          0
750009  T chỉ muốn nhảy vào cào nát bản ml của thằng đ...          0


--- 10 DÒNG CUỐI ---
                                                  content  sentiment
799990             

Batch 17: 100%|██████████| 50000/50000 [07:26<00:00, 111.97it/s]



=== ĐÃ LƯU XONG ĐẾN DÒNG: 850,000 ===
sentiment
1    21571
0    20139
2     8290
Name: count, dtype: int64


--- 10 DÒNG ĐẦU ---
                                                  content  sentiment
800000     Hay quá trời ơi xem đi xem lại phải 31 lần r í          0
800001        Dương tử  xinh quá .... như tiên nữ hạ phàm          1
800002  BÀ NỮ VÀ LỮ BỐ LÀ ĐỒ PHẢN TẶC TRUNG QUỐC SỐNG ...          0
800003                  Phượng Cửu vs Đế Quân của tui *•*          2
800004  Thùy Linh nhưng chứng cứ đó chưa được xác thực...          2
800005  hát dở, nhảy dở, nhạt nhẽo, ít nói là SJH. dựa...          0
800006                                      lữ bố láo nhỉ          0
800007                                      cảm ơn ae nha          1
800008                                  Nam chính xấu thế          0
800009                         trời ơi mong phim nhanh ra          0


--- 10 DÒNG CUỐI ---
                                      content  sentiment
849990                        s

Batch 18: 100%|██████████| 30009/30009 [04:23<00:00, 113.79it/s]



=== ĐÃ LƯU XONG ĐẾN DÒNG: 880,009 ===
sentiment
1    13391
0    11720
2     4898
Name: count, dtype: int64


--- 10 DÒNG ĐẦU ---
                                                  content  sentiment
850000  97 là tên trong vai phim cá mật hầm mực á bạn ...          1
850001  xem xong fim này không còn muốn xem fim nào nữ...          0
850002                co ai biết fim nao hay chỉ mình với          2
850003  bao h coi fim cũng rất ghét kiủ phụ huynh fản ...          0
850004                    tuyến giữa MU thua hoàn toàn AC          0
850005                              milan đá hay vch luôn          0
850006                              Minh lan truyện nhé          1
850007  Phim rất hay, ít có kịch bản phim nào khai thá...          1
850008                                        cảm ơn nhaa          1
850009      Ông của Bell chính là lão Zeus hóa thân xuống          0


--- 10 DÒNG CUỐI ---
                                                  content  sentiment
879999             

In [ ]:
final = pd.read_csv("fpt_full_sentiment.csv")

In [3]:
print("comment sau phân tích:", len(final))

comment sau phân tích: 880009


In [10]:
final['sentiment'].value_counts().reset_index()

,sentiment,count
0,0,370170
1,1,361503
2,2,148336
